# Original code

In [1]:
import gamdpy as gp
import h5py

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os.path

gp.select_gpu()

# Specify statepoint
num_part = 8000
rho = 1.200
temperature = 5.00
qA = 0.5
qB = -4*qA # (4:1 mixture)

# Setup configuration: 
configuration = gp.Configuration(D=3, compute_flags={'U': True, 'K': True})
configuration.make_positions(N=num_part, rho=rho)
configuration['m'] = 1.0 # Specify all masses to unity 
configuration.randomize_velocities(temperature=temperature, seed=1) # Initial high temperature for randomizing
configuration.ptype[::5] = 1 # Every fifth particle set to type 1 (4:1 mixture)

# Setup pair potential: Binary Kob-Andersen LJ mixture + coulomb.
LJ_Coulomb = gp.add_potential_functions(gp.LJ_12_6_sigma_epsilon, gp.make_IPL_n(1,first_parameter=2))
pair_func = gp.apply_shifted_force_cutoff(LJ_Coulomb)
sig = [[1.00, 0.80],
       [0.80, 0.88]]
eps = [[1.00, 1.50],
       [1.50, 0.50]]
qq = [[ qA*qA, qA*qB],
      [ qB*qA, qB*qB]]
cut = [[ 6, 6],
       [ 6, 6]]
pair_pot = gp.PairPotential(pair_func, params=[sig, eps, qq, cut], max_num_nbs=2000)

dt = 0.004  # timestep
num_timeblocks = 8           # Do simulation in this many 'blocks'. 
steps_per_timeblock = 1*1024  # ... each of this many steps

integrator = gp.integrators.NVT(temperature=temperature, tau=0.2, dt=dt)

# Setup runtime actions, i.e. actions performed during simulation of timeblocks
runtime_actions = [gp.TrajectorySaver(), gp.MomentumReset(100)]

compute_plan = gp.get_default_compute_plan(configuration)
compute_plan['nblist'] = 'N squared'
print(compute_plan)
sim = gp.Simulation(configuration, pair_pot, integrator, runtime_actions, 
                    num_timeblocks=num_timeblocks, steps_per_timeblock=steps_per_timeblock,
                    compute_plan=compute_plan, storage="memory") 

for block in sim.run_timeblocks():
    print(f'{sim.status(per_particle=True)}')
print(sim.summary())

Using device 0
{'pb': 32, 'tp': 2, 'skin': np.float32(1.0), 'UtilizeNIII': False, 'gridsync': True, 'nblist': 'N squared'}
timeblock= 0     time= 4.096       U= -3.134    W= 27.306    K= 7.609     
timeblock= 1     time= 8.192       U= -3.177    W= 27.117    K= 7.455     
timeblock= 2     time= 12.288      U= -3.133    W= 27.274    K= 7.498     
timeblock= 3     time= 16.384      U= -3.102    W= 27.358    K= 7.442     
timeblock= 4     time= 20.480      U= -3.198    W= 27.007    K= 7.578     
timeblock= 5     time= 24.576      U= -3.195    W= 27.035    K= 7.517     
timeblock= 6     time= 28.672      U= -3.137    W= 27.266    K= 7.399     
timeblock= 7     time= 32.768      U= -3.160    W= 27.212    K= 7.496     
Particles : 8000 
Steps : 8 * 1024 = 8_192 
Total run time  (incl. time spent between timeblocks): 9.44 s ( TPS: 8.68e+02 )
Simulation time (excl. time spent between timeblocks): 8.83 s ( TPS: 9.28e+02 )



In [2]:
sim.output['trajectory/positions'][-1, :, :, :]

array([[[-9.043234  ,  6.911984  , -6.9343834 ],
        [-8.730075  , -6.4342422 , -8.560802  ],
        [ 7.0227423 , -7.511242  ,  5.4511356 ],
        ...,
        [ 6.4945235 ,  9.246047  ,  6.570794  ],
        [-5.4631567 ,  8.180028  , -5.784579  ],
        [-9.000143  , -0.9728954 ,  8.022644  ]],

       [[-9.056675  ,  6.8977656 , -6.9437    ],
        [-8.728012  , -6.4294496 , -8.557322  ],
        [ 7.039266  , -7.5165586 ,  5.4532166 ],
        ...,
        [ 6.4836693 ,  9.237772  ,  6.5750933 ],
        [-5.4647884 ,  8.174968  , -5.791702  ],
        [-8.990078  , -0.96962595,  8.021738  ]],

       [[-9.070125  ,  6.8825088 , -6.9526906 ],
        [-8.725011  , -6.423395  , -8.552903  ],
        [ 7.055319  , -7.5219073 ,  5.4560943 ],
        ...,
        [ 6.4766507 ,  9.229281  ,  6.581971  ],
        [-5.466092  ,  8.169469  , -5.800543  ],
        [-8.978765  , -0.9643678 ,  8.021733  ]],

       ...,

       [[ 9.251301  ,  6.583805  , -7.1494465 ],
        [-8

In [3]:
gp.tools.print_h5_structure(sim.output)

initial_configuration/ (Group)
    ptype  (Dataset, shape=(8000,), dtype=int32)
    r_im  (Dataset, shape=(8000, 3), dtype=int32)
    scalars  (Dataset, shape=(8000, 4), dtype=float32)
    topology/ (Group)
        angles  (Dataset, shape=(0,), dtype=int32)
        bonds  (Dataset, shape=(0,), dtype=int32)
        dihedrals  (Dataset, shape=(0,), dtype=int32)
        molecules/ (Group)
    vectors  (Dataset, shape=(3, 8000, 3), dtype=float32)
trajectory/ (Group)
    images  (Dataset, shape=(8, 12, 8000, 3), dtype=int32)
    positions  (Dataset, shape=(8, 12, 8000, 3), dtype=float32)
    ptypes  (Dataset, shape=(8, 12, 8000), dtype=int32)
    steps  (Dataset, shape=(12,), dtype=int32)
    topologies/ (Group)
        block0000/ (Group)
            angles  (Dataset, shape=(0,), dtype=int32)
            bonds  (Dataset, shape=(0,), dtype=int32)
            dihedrals  (Dataset, shape=(0,), dtype=int32)
            molecules/ (Group)
        block0001/ (Group)
            angles  (Dataset, s

# New stuff

In [ ]:
# Setup configuration: 
c2 = gp.Configuration(D=3, compute_flags={'U': True, 'K': True})
c2.make_positions(N=num_part, rho=rho)
c2['m'] = 1.0 # Specify all masses to unity 
c2.randomize_velocities(temperature=temperature, seed=1) # Initial high temperature for randomizing
c2.ptype[::5] = 1 # Every fifth particle set to type 1 (4:1 mixture)

# Setup pair potential: Binary Kob-Andersen LJ mixture + coulomb.
LJ = gp.apply_shifted_force_cutoff(gp.LJ_12_6_sigma_epsilon)
LJ_pair = gp.PairPotential(LJ, params=[sig, eps, cut], max_num_nbs=2000)
Coulomb = gp.Electrostatics(charges=[qA, qB], cutoff=cut)

integrator = gp.integrators.NVT(temperature=temperature, tau=0.2, dt=dt)

# Setup runtime actions, i.e. actions performed during simulation of timeblocks
runtime_actions = [gp.TrajectorySaver(), gp.MomentumReset(100)]

compute_plan = gp.get_default_compute_plan(configuration)

sim2 = gp.Simulation(c2, [LJ_pair, Coulomb], integrator, runtime_actions, 
                    num_timeblocks=num_timeblocks, steps_per_timeblock=steps_per_timeblock,
                    compute_plan=compute_plan, storage="memory") 

for block in sim2.run_timeblocks():
    print(f'{sim2.status(per_particle=True)}')
print(sim2.summary())

timeblock= 0     time= 4.096       U= -3.164    W= 27.187    K= 7.402     
timeblock= 1     time= 8.192       U= -3.107    W= 27.464    K= 7.659     
